# 📘 学习注释版：Silver ERP Location

**Input：** `workspace.bronze.erp_loc_a101`  
**Output：** `workspace.silver.erp_customer_location`

主要学习：
- `AW-00011000` → `AW00011000`
- 国家代码统一
- 跨系统 Join Key 标准化


#Initialization

## 🧰 学习说明：导入 PySpark 函数/类型

这里仅准备后续清洗需要的 API，例如：
- `col()`：引用列
- `trim()`：去前后空格
- `StringType`：判断字符串类型
- `F.when()`：类似 SQL `CASE WHEN`

这一 Cell 不改变数据。


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import trim, col

# Read Bronze table

## 📖 学习说明：读取 Bronze Table

**Input：** `workspace.bronze.erp_loc_a101`  
**Process：** `spark.table()` 把 Catalog 中的 Table 读取成 DataFrame  
**Output：** 变量 `df`

后续所有 Silver 清洗都在这个 DataFrame 上进行。


In [0]:
df = spark.table("workspace.bronze.erp_loc_a101")

#Silver Transformations

##Trimming

## ✂️ 学习说明：批量 Trim 字符串

遍历 DataFrame 所有字段：

```text
如果字段类型 = String
→ trim()
→ 去掉前后空格
```

真实数据中 `" Jon"` 和 `"Jon"` 在比较/JOIN 时可能被认为不同，所以 Silver 要先清理。


In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

##Customer ID Cleanup

## 🔗 学习说明：统一 Location Customer Key

```text
AW-00011000
→ 删除 "-"
→ AW00011000
```

最终与 CRM / ERP Customer 使用相同格式。


In [0]:
df = df.withColumn("cid", F.regexp_replace(col("cid"), "-", ""))

##Country Normalization

## 🌍 学习说明：国家名称标准化

例如：

```text
DE       → Germany
US / USA → United States
空值     → n/a
```

统一 Code 后，Gold 不需要理解每个源系统自己的编码规则。


In [0]:
df = df.withColumn(
    "cntry",
    F.when(col("cntry") == "DE", "Germany")
     .when(col("cntry").isin("US", "USA"), "United States")
     .when((col("cntry") == "") | col("cntry").isNull(), "n/a")
     .otherwise(col("cntry"))
)

## Renaming Columns

## 🏷️ 学习说明：统一字段名称

把源系统字段名统一成业务更容易理解的名字。

例如：

```text
cst_id        → customer_id
cst_key       → customer_number
prd_nm        → product_name
```

意义：Silver 不只是“清洗值”，也统一数据模型/字段契约。


In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "cntry": "country"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity checks of dataframe

## 👀 学习说明：DataFrame Sanity Check

只显示前 10 行，快速确认当前 DataFrame：
- 字段是否正确
- 清洗是否生效
- 数据是否仍然存在

这一步不写表，只是开发时的中间检查。


In [0]:
df.limit(10).display()

#Writing Silver Table

## 💾 学习说明：把 DataFrame 持久化为 Delta Table

**Input：** 当前 `df`  
**Process：**
- `mode("overwrite")`：目标已存在时覆盖
- `format("delta")`：使用 Delta 格式
- `saveAsTable()`：注册为 Catalog Table

**Output：** `workspace.silver.erp_customer_location`

注意：这也是为什么 Bootcamp 可以重复运行而通常不会因为“表已存在”直接失败。


In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customer_location")

## Sanity checks of silver table

## ✅ 学习说明：验证 ERP Location Silver 表

重点检查：
- Customer Key 中的 `-` 是否删除
- Country 是否统一成标准名称


In [0]:
%sql
SELECT * FROM workspace.silver.erp_customer_location LIMIT 10